In [ ]:
import os
import requests
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import cv2
import reverse_geocoder as rg
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupShuffleSplit

# ======================================================
# CONFIGURATION
# ======================================================
MAPILLARY_TOKEN = "MLY|26664131136518032|9af8051ce63bc8d3adb2abc9b889ed15"
SAVE_DIR = "mapillary_data"
BBOX = [west, south, east, north] # Define your coordinates
NUM_REGIONS = 1000
BATCH_SIZE = 32
LR = 1e-4
NUM_EPOCHS = 20

# ======================================================
# 1. DATA ACQUISITION
# ======================================================

def fetch_images_in_bbox(west, south, east, north, token, max_images=5000):
    os.makedirs(SAVE_DIR, exist_ok=True)
    url = "https://graph.mapillary.com/images"
    params = {
        "access_token": token,
        "bbox": f"{west},{south},{east},{north}",
        "fields": "id,geometry,thumb_1024_url",
        "limit": 100 
    }
    
    records = []
    while url and len(records) < max_images:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json().get("data", [])
        if not data: break
            
        for item in data:
            if len(records) >= max_images: break
            img_id, img_url = item["id"], item.get("thumb_1024_url")
            coords = item.get("geometry", {}).get("coordinates") 
            
            if img_url and coords:
                path = os.path.join(SAVE_DIR, f"{img_id}.jpg")
                if not os.path.exists(path):
                    try:
                        img_data = requests.get(img_url, timeout=10).content
                        with open(path, "wb") as f: f.write(img_data)
                    except Exception: continue
                records.append({"image_path": path, "lat": coords[1], "lon": coords[0], "id": img_id})
        
        link_header = response.headers.get("Link")
        url, params = None, None
        if link_header and 'rel="next"' in link_header:
            links = link_header.split(',')
            for link in links:
                if 'rel="next"' in link:
                    url = link[link.find("<")+1:link.find(">")]
                    break
    
    df = pd.DataFrame(records)
    df.to_csv("raw_data.csv", index=False)
    return df

# ======================================================
# 2. GEOGRAPHIC UTILITIES
# ======================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    $d = 2R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos\phi_1\cos\phi_2\sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)$
    """
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlambda = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def latlon_to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.stack([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)], axis=1)

def generate_voronoi_regions(coords, k):
    xyz = latlon_to_xyz(coords[:,0], coords[:,1])
    kmeans = KMeans(n_clusters=min(k, len(coords)), random_state=42, n_init=10)
    labels = kmeans.fit_predict(xyz)
    centroids_xyz = kmeans.cluster_centers_
    lat = np.degrees(np.arcsin(centroids_xyz[:,2]))
    lon = np.degrees(np.arctan2(centroids_xyz[:,1], centroids_xyz[:,0]))
    return np.stack([lat, lon], axis=1), labels

def add_country_labels(df):
    coords = list(zip(df['lat'], df['lon']))
    results = rg.search(coords)
    df['country_code'] = [r['cc'] for r in results]
    df['country_idx'] = pd.Categorical(df['country_code']).codes
    return df

# ======================================================
# 3. DATASET
# ======================================================

class MapillaryDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        normalize = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

        if self.is_train:
            self.transform = T.Compose([
                T.RandomResizedCrop(224, scale=(0.7, 1.0)),
                T.ColorJitter(0.3, 0.3, 0.3, 0.1),
                T.RandomHorizontalFlip(),
                T.ToTensor(),
                normalize,
                T.RandomErasing(p=0.3, scale=(0.02, 0.2)) 
            ])
        else:
            self.transform = T.Compose([T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(), normalize])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.transform(Image.open(row['image_path']).convert('RGB'))
        return (img, 
                torch.tensor(row['country_idx']).long(), 
                torch.tensor(row['region_idx']).long(), 
                torch.tensor([row['lat'], row['lon']], dtype=torch.float32))

# ======================================================
# 4. HIERARCHICAL MODEL & GRAD-CAM
# ======================================================



class GeoGuessrBot(nn.Module):
    def __init__(self, num_countries, num_regions):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.avgpool = backbone.avgpool
        dim = backbone.fc.in_features
        
        self.country_head = nn.Linear(dim, num_countries)
        self.region_head = nn.Linear(dim, num_regions)
        self.gradients = None

    def activations_hook(self, grad): self.gradients = grad

    def forward(self, x):
        h = self.features(x)
        if x.requires_grad: h.register_hook(self.activations_hook)
        pooled = torch.flatten(self.avgpool(h), 1)
        return self.country_head(pooled), self.region_head(pooled), h

    def get_activations_gradient(self): return self.gradients

def generate_gradcam(model, img_tensor, pred_idx):
    model.zero_grad()
    c_logits, r_logits, features = model(img_tensor.unsqueeze(0))
    r_logits[0, pred_idx].backward()
    
    grads = model.get_activations_gradient()
    weights = torch.mean(grads, dim=[0, 2, 3])
    activations = features.detach()
    for i in range(activations.size(1)): activations[:, i, :, :] *= weights[i]
        
    heatmap = torch.mean(activations, dim=1).squeeze()
    heatmap = np.maximum(heatmap.cpu(), 0)
    return heatmap / (torch.max(heatmap) + 1e-10)

# ======================================================
# 5. TRAINING & EVALUATION
# ======================================================



def evaluate(model, loader, centroids, device):
    model.eval()
    errors = []
    with torch.no_grad():
        for imgs, _, _, true_coords in loader:
            _, r_logits, _ = model(imgs.to(device))
            preds = torch.argmax(r_logits, dim=1).cpu().numpy()
            true_coords = true_coords.numpy()
            for i, k in enumerate(preds):
                dist = haversine_distance(true_coords[i,0], true_coords[i,1], centroids[k,0], centroids[k,1])
                errors.append(dist)
    return np.median(errors)

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if not os.path.exists("raw_data.csv"):
        metadata = fetch_images_in_bbox(BBOX[0], BBOX[1], BBOX[2], BBOX[3], MAPILLARY_TOKEN)
    else:
        metadata = pd.read_csv("raw_data.csv")

    metadata = add_country_labels(metadata)
    centroids, region_labels = generate_voronoi_regions(metadata[['lat', 'lon']].values, NUM_REGIONS)
    metadata['region_idx'] = region_labels

    splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, val_idx = next(splitter.split(metadata, groups=metadata.region_idx))
    
    train_loader = DataLoader(MapillaryDataset(metadata.iloc[train_idx], True), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(MapillaryDataset(metadata.iloc[val_idx], False), batch_size=BATCH_SIZE)

    model = GeoGuessrBot(metadata.country_idx.nunique(), len(centroids)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(NUM_EPOCHS):
        model.train()
        for imgs, c_targets, r_targets, _ in train_loader:
            imgs, c_targets, r_targets = imgs.to(device), c_targets.to(device), r_targets.to(device)
            optimizer.zero_grad()
            c_out, r_out, _ = model(imgs)
            loss = criterion(c_out, c_targets) + criterion(r_out, r_targets)
            loss.backward()
            optimizer.step()
        
        median_error = evaluate(model, val_loader, centroids, device)
        print(f"Epoch {epoch+1} | Median Error: {median_error:.2f} km")

    # Final Interpretability Check
    
    
if __name__ == "__main__":
    main()